Imports

In [15]:
import re
import requests
import geocoder
import time
from openai import OpenAI

connectie LM studio

In [16]:
client = OpenAI(
base_url = "http://127.0.0.1:1234/v1/chat/completions",
api_key="lm-studio"
)

model inladen

In [17]:
model_name = "nvidia/nemotron-3-nano-4b"

lijst maken van ingredienten op voorraad

In [18]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

locatie opvragen

In [19]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

Promptstructuur

In [20]:
prompts = [
    f"""
Geef exact 5 gerechten op basis van deze ingrediënten: {items}.

vaste opzet:
gerecht nr|gerecht|ingredienten|duur|nieuwe ingredienten|

Voor elk gerecht:
- bereidingstijd <= 30 minuten
- extra ingrediënt
- en zet letterlijk dit aan het einde van de text zodat ik hem er later uit kan halen: 'nieuw ingredient = (ingredient)'
"""
]

Startijd

In [21]:
for prompt in prompts:
    start_time = time.time()

Data opslaan

In [22]:
data = {
    "model": model_name,
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


antwoord opvragen

In [ ]:
try:
    response = requests.post(client.base_url, json=data)
    response.raise_for_status()

    end_time = time.time()

    result = response.json()

    answer = result["choices"][0]["message"]["content"]
    model_name = result.get("model", "onbekend")
    response_length = len(answer)

    print(answer)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

except requests.exceptions.RequestException as e:
    print(f"Fout bij API-aanroep: {e}")


1|Aardappel en karnofstokensoep met appelmoes|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (zucchini)  
2|Karnof en aardappel stoffrit met appelmoes en komkommer|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (komkommer)  
3|Appelmoes kipgratin met aardappel en paprika|aardappel, appelmoes, paprika|28 min|nieuw ingredient = (paprika)  
4|Aardappel, karnof en appelmoes salade met basilicum|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (basilicum)  
5|Karnofstokensoep met appelmoes en citroen|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (citeroen)

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 601 karakters
Tijd: 63.52 seconden
--------------------------------------------------


???

In [24]:
for answe in answer.splitlines():
    match = re.search(r"nieuw ingrediënt\s*=\s*\((.*?)\)", answe)

    if match:
        ingredient = match.group(1)
        url = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
        answe += f"|{url}"

    print(answe)



1|Aardappel en karnofstokensoep met appelmoes|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (zucchini)  
2|Karnof en aardappel stoffrit met appelmoes en komkommer|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (komkommer)  
3|Appelmoes kipgratin met aardappel en paprika|aardappel, appelmoes, paprika|28 min|nieuw ingredient = (paprika)  
4|Aardappel, karnof en appelmoes salade met basilicum|aardappel, karbonade, appelmoes|25 min|nieuw ingredient = (basilicum)  
5|Karnofstokensoep met appelmoes en citroen|aardappel, karbonade, appelmoes|30 min|nieuw ingredient = (citeroen)
